# AMReX: adaptive mesh refinement

Goal: Run a 1-D dam break on AMReX with a refinement level that follows the waves and read the cells per level.

Prerequisite: [AMReX: from symbolic model to generated C++](minimal.ipynb)

## 1. The case

The model and the numerics are those of the [NumPy tutorial](../swe/simple_numpy.ipynb). The mesh is level 0.

In [1]:
import numpy as np

import zoomy_core.model.boundary_conditions as BC
import zoomy_core.model.initial_conditions as IC
from zoomy_core.mesh import LSQMesh
from zoomy_core.model.models import SME
from zoomy_core.model.models.closures import ManningFriction, Newtonian, StressFree
from zoomy_core.numerics import Limiter, NumericalSystemModel, ReconstructionSpec
from zoomy_core.systemmodel import SystemModel

model = SME(
    level=0, dimension=2,
    closures=[Newtonian(), ManningFriction(), StressFree()],
    boundary_conditions=BC.BoundaryConditions(
        [BC.Extrapolation(tag="left"), BC.Extrapolation(tag="right")]),
)
sm = SystemModel.from_model(model)
sm.initial_conditions = IC.UserFunction(
    function=lambda x: np.array([0.0, 2.0 if x[0] < 5.0 else 1.0, 0.0]))
nsm = NumericalSystemModel.from_system_model(
    sm, reconstruction=ReconstructionSpec(order=2, limiter=Limiter.minmod))
mesh = LSQMesh.create_1d(domain=(0.0, 10.0), n_inner_cells=128)
n = mesh.n_inner_cells
Q0 = sm.initial_conditions.apply(np.asarray(mesh.cell_centers)[:, :n], np.zeros((3, n)))

## 2. Refine where the depth changes

`amr` is a dict. Level 1 (`ratio` 2) covers the cells where $|\nabla h|$ exceeds 0.1, regridded every 4 steps. The cell counts of the mesh given to `run` must be divisible by `blocking_factor` (8).

In [2]:
import tempfile
from pathlib import Path

from zoomy_amrex.symsolver import SymbolicHyperbolicSolverAmrex

solver = SymbolicHyperbolicSolverAmrex(
    time_end=0.5, cfl=0.9,
    amr=dict(max_level=1, ratio=2, regrid_int=4, tag=("h", "gradient", 0.1)))
time, Q = solver.run(mesh, nsm, Q0, Path(tempfile.mkdtemp()) / "dambreak_amr")   # Q: level 0, averaged down

h = np.asarray(Q)[1]
cells = [int(line.split()[-1]) for line in solver.log.splitlines() if "AMR: level" in line]
print(f"t = {time:.3f}, h in [{h.min():.5f}, {h.max():.5f}], mass = {h.sum() * 10.0 / h.size:.6f}")
print(f"cells per level at the end: {cells}")

t = 0.500, h in [1.00000, 2.00000], mass = 15.000000
cells per level at the end: [128, 24]


In [3]:
assert len(cells) == 2 and cells[1] > 0, "no refined level"
assert abs(h.sum() * 10.0 / h.size - 15.0) < 1e-9, "mass not conserved"